# Module 1 : 70 ans d'IA en 30 minutes

*Mardi 23 juin 2026, 9h-12h · Intervenant : Corentin Vande Kerckhove*

**CONSEIL :** Commencez par enregistrer une copie de ce notebook dans votre Google Drive (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes !

**Objectifs d'apprentissage**
- Situer l'IA générative dans l'histoire de l'IA (symbolique → ML → DL → Transformers → LLM)
- Voir la **même tâche** (classer un texte par parti politique) traverser chaque époque
- Comprendre ce que chaque génération ajoute à la précédente
- Voir **un modèle apprendre** depuis des données, et lire ce qu'il a appris
- Apprivoiser un premier appel à un grand modèle de langage

**Pré-requis :** aucun, c'est le premier module de la formation.
**Paquets requis :** `scikit-learn`, `transformers`, `openai`, `pandas`

## Running example : un peu de politique française

On va se balader dans 70 ans d'IA avec une question fil rouge : **« à quel parti appartient l'auteur de cette déclaration ? »**.

Le RN aime parler d'immigration. LFI préfère les salaires et le peuple. LR adore le marché libre et les entreprises. EELV milite pour la transition écologique et le climat. Vous voyez l'idée.

> Pour illustrer les concepts, on travaille sur un **mini-corpus de 18 phrases inventées** : 16 bien marquées (4 par parti) et 2 volontairement **ambiguës**. C'est volontairement petit : assez pour voir 70 ans d'IA défiler, et sans manipuler de données personnelles réelles.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-opener.jpg" width="440" alt="Pions de couleur et urne de vote">

*Fil rouge du module : à quel parti appartient une déclaration ? Pions de couleur devant une urne de vote.*

## Setup

In [ ]:
# Sur Colab : exécute cette cellule pour installer les paquets (déjà installés sur un poste local)
!pip install -q transformers openai

import pandas as pd

In [ ]:
# 16 déclarations bien marquées (4 par parti) + 2 cas AMBIGUS où le mot-clé pointe vers le mauvais parti
data = pd.DataFrame({
    "texte": [
        # RN
        "Il faut réduire l'immigration et défendre nos frontières.",
        "L'identité française doit être protégée.",
        "Sécurité, frontières, immigration : nos priorités sont claires.",
        "Notre identité et nos traditions face à la submersion migratoire.",
        # LFI
        "Augmentons les salaires et taxons les plus riches.",
        "Le peuple souffre pendant que la finance s'enrichit.",
        "Une révolution citoyenne contre l'oligarchie est nécessaire.",
        "Taxons les riches pour financer les services publics et les salaires.",
        # LR
        "Le marché libre et la baisse des charges relanceront l'économie.",
        "Soutenons les entreprises et libérons les énergies.",
        "Moins d'État, plus de responsabilité et de liberté économique.",
        "Baissons les charges des entreprises pour relancer le marché du travail.",
        # EELV
        "La transition écologique est urgente et juste.",
        "Le climat et la biodiversité au cœur des politiques publiques.",
        "Sortons des énergies fossiles, investissons dans le renouvelable.",
        "Une transition écologique juste pour préserver le climat.",
        # Cas ambigus : un parti emprunte le mot-clé d'un autre
        "Une économie de marché plus sobre et respectueuse du vivant.",        # EELV, mais dit « marché » (LR)
        "Notre transition doit rester pragmatique et favorable aux patrons.",   # LR, mais dit « transition » (EELV)
    ],
    "parti": ["RN", "RN", "RN", "RN",
              "LFI", "LFI", "LFI", "LFI",
              "LR", "LR", "LR", "LR",
              "EELV", "EELV", "EELV", "EELV",
              "EELV", "LR"],
})
data

## 1.1 IA symbolique (années 50-80)

Nos ancêtres en IA n'avaient ni Internet, ni puissance de calcul, ni montagnes de données.
Leur arme : **des règles écrites à la main** par un expert humain.

Concrètement : pour reconnaître un texte du RN, on dresse une courte liste de mots qui reviennent souvent chez eux (« immigration », « frontière »…). Idem pour les autres partis. Puis on compte, pour chaque texte, combien de mots-clés de chaque parti apparaissent. Le parti qui en a le plus gagne. Voilà notre première « IA ».

**Force** : c'est lisible, on sait exactement ce que le programme fait.
**Limite** : c'est très fragile. Un mot manquant, un mot emprunté à un autre parti, et tout casse.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-symbolic.png" width="560" alt="Schéma IA symbolique">

*L'humain écrit des règles de mots-clés ; la machine se contente de compter.*

In [ ]:
import random

# Règles écrites à la main : quelques mots-clés par parti (liste volontairement courte)
RULES = {
    "RN":   ["immigration", "frontière", "identité"],
    "LFI":  ["salaire", "riche", "peuple"],
    "LR":   ["marché", "entreprise", "charges"],
    "EELV": ["climat", "transition", "écolog"],
}

def count_keywords(text):
    """Pour chaque parti, compter combien de ses mots-clés apparaissent dans le texte."""
    text = text.lower()
    return {party: sum(kw in text for kw in words) for party, words in RULES.items()}

def predict_by_rules(text, rng=random.Random(0)):
    """Parti au plus grand score. Égalité -> tirage au hasard. Aucun mot-clé -> '?'."""
    scores = count_keywords(text)
    best = max(scores.values())
    if best == 0:
        return "?"                      # aucun mot-clé reconnu
    winners = [party for party, s in scores.items() if s == best]
    return rng.choice(winners)          # en cas d'égalité, on tranche au hasard

data["pred_rules"] = data["texte"].apply(predict_by_rules)
data[["parti", "texte", "pred_rules"]]

### Mesurer les erreurs

Combien de prédictions sont **correctes**, **fausses**, ou **sans réponse** (`?`, aucun mot-clé reconnu) ?

In [ ]:
correct = (data["parti"] == data["pred_rules"]).sum()
unknown = (data["pred_rules"] == "?").sum()
wrong   = len(data) - correct - unknown
print(f"Sur {len(data)} phrases : {correct} correctes, {wrong} fausses, {unknown} sans réponse (?)")
print(f"Exactitude des règles : {correct / len(data):.0%}")

# Égalité de mots-clés -> on tranche au hasard (donc instable) :
print("\nExemple d'égalité :", predict_by_rules("Réduire l'immigration tout en augmentant les salaires."))

Les règles écrites à la main **se trompent** : certaines phrases n'ont aucun mot-clé de notre liste (`?`), et les deux phrases ambiguës partent vers le mauvais parti.

### Hack Time

Complétez le dictionnaire `RULES` (ajoutez les mots-clés manquants) pour corriger un **maximum** d'erreurs, puis relancez la mesure. Jusqu'où descendez-vous ? Les deux phrases ambiguës, elles, résistent : pourquoi ?

In [ ]:
# Votre code ici

## 1.2 Machine Learning statistique (années 90-2010)

L'idée centrale du ML : **arrêter d'écrire les règles à la main, et les laisser apprendre depuis les données**.

Reformulons : en 1.1, c'est nous qui décidions que tel mot compte pour tel parti. Le programme se contentait d'appliquer notre liste, il n'apprenait rien tout seul.

En machine learning, on fait l'inverse, en trois temps :
1. On rassemble des **exemples déjà classés** : nos 18 phrases, chacune avec son parti.
2. On les présente à la machine sous une forme qu'elle peut traiter (ici, pour chaque phrase : quels mots-clés sont présents).
3. La machine **repère toute seule les régularités** : quels mots annoncent quel parti.

Retiens l'essentiel : **au lieu de lui dicter les règles, on lui montre des exemples et elle les déduit**. Le vocabulaire technique et le détail des modèles viendront au module 2.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-ml.png" width="560" alt="Schéma machine learning">

*On ne lui écrit aucune règle : on lui montre des exemples déjà classés et elle apprend quels mots annoncent quel parti.*

### 1.2.1 Transformer les phrases en tableau de chiffres

On donne au modèle une liste de mots-clés **plus riche** que nos règles, et pour chaque mot une colonne `1` (présent) / `0` (absent).

In [ ]:
# Liste de mots-clés plus complète que les règles écrites à la main
KEYWORDS = ["immigration", "frontière", "identité", "migratoire",
            "salaire", "riche", "peuple", "oligarchie",
            "marché", "entreprise", "charges", "responsabilité",
            "climat", "transition", "écolog", "renouvelable"]

def extract_features(text):
    text = text.lower()
    return {kw: int(kw in text) for kw in KEYWORDS}

features = pd.DataFrame([extract_features(t) for t in data["texte"]])
features.insert(0, "parti", data["parti"])
features

Vous obtenez un **tableau** : chaque ligne est une phrase, chaque colonne un mot-clé, avec `1` si le mot est présent et `0` sinon. C'est sur ce tableau de 0 et de 1 que la machine va travailler.

### 1.2.2 Voir le modèle apprendre (et faire moins d'erreurs)

In [ ]:
from sklearn.linear_model import LogisticRegression

X = pd.DataFrame([extract_features(t) for t in data["texte"]])
y = data["parti"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print(f"Exactitude des règles écrites à la main : {correct / len(data):.0%}")
print(f"Exactitude de la régression logistique : {model.score(X, y):.0%}")

**Le machine learning fait le travail du Hack Time tout seul.** Sans qu'on touche aux règles, le modèle a « trouvé » dans les données les mots qui manquaient et corrigé une partie des erreurs. On peut même lire ce qu'il a appris : ses coefficients (un poids par mot-clé et par parti).

In [ ]:
coefs = pd.DataFrame(model.coef_, columns=KEYWORDS, index=model.classes_).round(2)
coefs

### 1.2.3 Faire prédire le modèle sur une phrase de votre cru

In [ ]:
ma_phrase = "Taxons les grandes fortunes pour augmenter les salaires et défendre le peuple."

mes_features = pd.DataFrame([extract_features(ma_phrase)])
prediction = model.predict(mes_features)[0]
probabilites = pd.Series(model.predict_proba(mes_features)[0], index=model.classes_).round(2)

print(f"Prédiction : {prediction}")
print()
print("Probabilités estimées par le modèle :")
print(probabilites)

### Hack Time

Modifiez `ma_phrase` pour faire prédire **chaque parti** au modèle, et regardez la confiance (les probabilités). Que se passe-t-il avec une phrase **sans aucun mot-clé** ?

In [ ]:
# Votre code ici

## 1.3 Deep Learning (2010-2017)

Bascule majeure : on **arrête de fabriquer les features à la main**.

Jusqu'ici, c'est nous qui écrivions la liste de mots-clés (`RULES`, puis `KEYWORDS`). Le deep learning part du **texte brut** : on lui donne **tous les mots** des phrases, et c'est le modèle qui décide lesquels comptent. Il fabrique ses propres features.

> **Pour voir un réseau de neurones apprendre en direct** : ouvrez [playground.tensorflow.org](https://playground.tensorflow.org/) dans un nouvel onglet, choisissez un dataset, cliquez sur Play, et regardez les couches « apprendre » à séparer les points. C'est la même mécanique, en beaucoup plus gros, sous le capot des LLMs.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-dl.png" width="560" alt="Schéma deep learning">

*Un réseau de neurones part du texte brut : ses couches cachées fabriquent les features à la place de l'humain.*

### 1.3.1 Un réseau de neurones sur le texte brut

On ne lui donne plus notre liste de mots-clés, mais **tous les mots** du corpus (un `CountVectorizer` les compte automatiquement). Le réseau (un *multilayer perceptron*) apprend ensuite à partir de ce vocabulaire complet.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPClassifier

# Aucune liste écrite à la main : on part de TOUS les mots des phrases.
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(data["texte"])
print(f"Le modèle fabrique ses features tout seul : "
      f"{len(vectorizer.get_feature_names_out())} mots détectés dans le corpus.")

mlp = MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=3000, random_state=3)
mlp.fit(X_text, y)
print(f"Exactitude du réseau de neurones : {mlp.score(X_text, y):.0%}")

Parce qu'il voit **tous les mots** (et pas seulement nos mots-clés), le réseau attrape même les phrases ambiguës : il repère que « sobre », « vivant » ou « patrons » trahissent le vrai parti, là où nos règles ne voyaient que « marché » ou « transition ».

### Hack Time

Faites varier la taille du réseau (`hidden_layer_sizes`) et relancez l'entraînement :

- **Réduisez-le** à `(1,)`, un seul neurone caché : il n'a plus assez de capacité pour séparer les 4 partis et son exactitude **chute** nettement. (`(2,)` est déjà à la limite : le résultat dépend de l'initialisation.)
- **Agrandissez-le** à `(64, 32)` : l'exactitude repasse à 100%... mais sur seulement 18 phrases, c'est un modèle **bien trop complexe**. Il mémorise le corpus par cœur ; sur de vraies données, ce surdimensionnement le ferait **se dégrader sur des exemples nouveaux** (le surapprentissage, qu'on verra au module 2).

In [ ]:
# Votre code ici

## 1.4 Transformers et grands modèles de langage (2017→)

**2017 : *Attention is all you need***. Une nouvelle architecture, le **transformer**, débarque chez Google.

Pourquoi ça change tout ? Parce que pour la première fois, un modèle peut être pré-entraîné sur d'énormes quantités de texte, **comprendre le sens des mots dans leur contexte**, et être ensuite adapté à n'importe quelle tâche avec très peu d'exemples.

L'innovation clé : le **mécanisme d'attention**. Chaque mot regarde tous les autres pour saisir son rôle exact dans la phrase.

**Transformer et LLM, quel lien ?** Le *transformer* est l'**architecture** (le moteur). Quand on entraîne ce moteur à très grande échelle sur quasiment tout le texte du web, on obtient un **grand modèle de langage (LLM)** comme ChatGPT, Gemini ou Mistral. Ici, on teste d'abord un transformer en mode **classifieur** ; on verra sa version **générative** (le LLM) en 1.6.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-transformers.png" width="560" alt="Schéma mécanisme d'attention">

*L'attention : chaque mot fixe son sens en regardant les autres mots de la phrase.*

### 1.4.1 Charger un transformer pré-entraîné

On utilise un transformer **général**, déjà entraîné sur d'énormes corpus multilingues, **sans aucun entraînement** sur nos partis. C'est un usage **zero-shot** : on lui donne la liste des étiquettes possibles, il se débrouille avec ses connaissances générales du langage.

> Le **chargement** télécharge le modèle (30-60 s la première fois). On le met dans **sa propre cellule**, à ne lancer qu'une fois.

In [ ]:
from transformers import pipeline

# Téléchargement + chargement du modèle (peut prendre 30-60 s la première fois).
classifier = pipeline("zero-shot-classification",
                      model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

### 1.4.2 Le transformer classe nos phrases

On l'applique directement au **texte brut**, sans rien lui apprendre. C'est un peu **lent** : chaque phrase traverse un gros réseau de neurones.

In [ ]:
# Un transformer "zero-shot" comprend mieux des étiquettes DÉCRITES en langage naturel
# que des sigles (RN, LFI...). On décrit chaque parti, puis on retient le mieux noté.
LABELS = {
    "immigration, frontières et identité nationale": "RN",
    "salaires, justice sociale et impôt sur les riches": "LFI",
    "marché libre, entreprises et baisse des charges": "LR",
    "écologie, climat et transition": "EELV",
}

# zero-shot : aucune phrase d'entraînement. (Lent : chaque phrase passe dans un gros modèle.)
preds_transformer = [LABELS[classifier(t, candidate_labels=list(LABELS))["labels"][0]]
                     for t in data["texte"]]
accuracy = (pd.Series(preds_transformer, index=data.index) == data["parti"]).mean()
print(f"Exactitude du transformer, sans aucun entraînement sur nos phrases : {accuracy:.0%}")

Remarquable : **sans avoir vu une seule de nos phrases**, le transformer s'en sort très bien. Attention quand même à la comparaison : nos modèles entraînés (régression, réseau) ont **mémorisé** ces phrases. La vraie épreuve, c'est de tester tout le monde sur des phrases **nouvelles** (section 1.5).

### Hack Time

Remplacez les `LABELS` par des libellés plus parlants (par exemple `"extrême droite"`, `"gauche radicale"`, `"droite libérale"`, `"écologistes"`) et relancez la classification sur une phrase de test : la prédiction change-t-elle ?

In [ ]:
# Votre code ici

## 1.5 Comparaison des 4 modèles, sur des phrases nouvelles

Jusqu'ici, les modèles entraînés (régression, réseau) ont vu nos 18 phrases : facile pour eux de les reconnaître. **L'épreuve de vérité**, c'est de classer des phrases **jamais vues**, qui paraphrasent les thèmes des partis **sans réutiliser nos mots-clés**.

On teste les 4 générations sur un petit jeu **équilibré** (autant de phrases par parti), construit en **quatre étages** de difficulté croissante : mot-clé des règles → mot-clé connu de la seule régression → mot du corpus vu du seul réseau → synonyme que seul le transformer comprend. On regarde alors **où chaque modèle commence à caler**.

In [ ]:
nouvelles = pd.DataFrame({
    "texte": [
        "Augmentons les salaires et taxons les riches.",            # mot-clé connu des règles
        "Le marché libre et la baisse des charges.",                # mot-clé connu des règles
        "Face à la submersion migratoire, défendons-nous.",         # « migratoire » : connu de la régression, pas des règles
        "Misons sur le renouvelable plutôt que le fossile.",        # « renouvelable » : idem, régression mais pas règles
        "Une révolution citoyenne pour les services publics.",      # mots du corpus, hors mots-clés : seul le réseau les a vus
        "Libérons l'économie et soutenons les patrons.",            # idem, hors mots-clés
        "L'arrivée massive d'étrangers inquiète la population.",     # synonymes seulement : réservé au transformer
        "Protégeons la planète et ses écosystèmes fragiles.",       # synonymes seulement : réservé au transformer
    ],
    "parti_vrai": ["LFI", "LR", "RN", "EELV", "LFI", "LR", "RN", "EELV"],
})
nouvelles

### 1.5.1 Lancer les 4 modèles côte à côte

In [ ]:
feats_new = pd.DataFrame([extract_features(t) for t in nouvelles["texte"]])

# La régression ne connaît que ses mots-clés : sans aucun mot-clé actif, elle n'a
# rien pour décider, donc elle s'abstient ('?'), comme les règles.
def predire_regression(i):
    return model.predict(feats_new.iloc[[i]])[0] if feats_new.iloc[i].sum() > 0 else "?"

resultats = nouvelles.copy()
resultats["① règles"]      = resultats["texte"].apply(predict_by_rules)
resultats["② régression"]  = [predire_regression(i) for i in range(len(feats_new))]
resultats["③ réseau"]      = mlp.predict(vectorizer.transform(nouvelles["texte"]))
resultats["④ transformer"] = [LABELS[classifier(t, candidate_labels=list(LABELS))["labels"][0]]
                              for t in nouvelles["texte"]]
resultats

In [ ]:
# Exactitude de chaque modèle sur les phrases nouvelles
for col in ["① règles", "② régression", "③ réseau", "④ transformer"]:
    acc = (resultats[col] == resultats["parti_vrai"]).mean()
    print(f"{col:<16} : {acc:.0%}")

### 1.5.2 Lecture du tableau

Ce qui compte n'est pas le pourcentage exact (il bouge un peu selon les versions de librairies, sur un jeu aussi petit), mais l'**ordre** et la **lecture colonne par colonne** : à chaque génération, le modèle répond là où le précédent **calait**.

Le jeu de test est construit en **quatre étages** (deux phrases chacun, équilibrés entre partis) :

- **① Règles** : ne reconnaissent que leur courte liste écrite à la main. Elles ne passent que les phrases contenant un de ces mots ; partout ailleurs, `?`. Le plancher.
- **② Régression** : même logique, mais un vocabulaire de mots-clés **plus riche** (elle connaît « migratoire », « renouvelable »…). Elle gagne ces phrases ; sans mot-clé, elle n'a rien pour décider et **s'abstient** elle aussi (`?`).
- **③ Réseau** : il a vu **tous les mots** du corpus, pas seulement nos mots-clés → il répond encore sur « services publics », « patrons »… là où la régression s'abstenait. Mais il **bute sur les synonymes** jamais vus à l'entraînement.
- **④ Transformer** : il **comprend le sens** indépendamment des mots exacts → il gère même les synonymes (« étrangers » ≈ immigration, « écosystèmes » ≈ écologie). Le haut de l'échelle.

L'ordre **transformer ≥ réseau ≥ régression ≥ règles** tient parce que chaque modèle **récupère les phrases que le précédent ratait**, sans en perdre. C'est cette capacité croissante à saisir le **sens** (au-delà des mots exacts) qui a ouvert la voie à l'IA générative (section 1.6).

## 1.6 IA générative (2020 → aujourd'hui)

Le transformer de 1.4 « comprend » le texte. Pour qu'il **génère** du texte, on l'utilise dans une variante **décodeur** (le modèle ne voit que les mots à sa gauche et apprend à prédire le suivant). Entraîné à très grande échelle, c'est le **LLM** dont on parlait en 1.4.

C'est ce qui a donné ChatGPT, Gemini, Mistral : le modèle ne se contente plus de classer, il **génère sa réponse en langage naturel** et peut expliquer son choix.

Magique ? Pas vraiment. Sous le capot, c'est toujours du transformer. Mais la combinaison **échelle massive + affinage par retours humains** rend le résultat saisissant.

> Note : on dialogue avec un LLM via une **API**, un service en ligne auquel on envoie du texte et qui renvoie une réponse. Pour s'authentifier, on utilise une « clé d'API ». Une clé vous sera **communiquée en classe** : il suffit de la coller dans la variable `api_key` de la cellule ci-dessous, puis de relancer la cellule.

> **Esprit critique.** La réponse d'un LLM n'est pas une source de vérité : il peut halluciner, il n'est pas parfaitement reproductible, et ses sorties doivent être validées, comme tout instrument de recherche.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j2/1_matin/img/m1/m1-generative.png" width="560" alt="Schéma génération mot à mot">

*Un modèle génératif prédit le mot suivant selon des probabilités, puis recommence.*

In [ ]:
from openai import OpenAI

# Colle ici la clé OpenAI communiquée en classe (entre les guillemets) :
api_key = "COLLE_TA_CLE_ICI"

if not api_key or api_key == "COLLE_TA_CLE_ICI":
    print("Aucune clé renseignée.")
    print("Colle la clé OpenAI communiquée en classe dans la variable api_key ci-dessus,")
    print("puis relance cette cellule pour interroger le modèle en direct.")
else:
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": "Classe cette déclaration parmi {RN, LFI, LR, EELV} "
                       "et justifie en une phrase : "
                       "'Augmentons les salaires et taxons les plus riches.'"
        }],
    )
    print(response.choices[0].message.content)

### Hack Time

Donnez au modèle génératif les **phrases nouvelles** de la section 1.5 (`nouvelles["texte"]`) et demandez-lui de classer chacune parmi {RN, LFI, LR, EELV}. Compare ses réponses à celles du transformer : qui s'en sort le mieux sur les synonymes ?

In [ ]:
# Votre code ici
# Indice : boucle sur nouvelles["texte"] et réutilise le client OpenAI ci-dessus.

## Récap module 1

| Section | Approche | Entrée | Ce qu'elle apporte |
|---|---|---|---|
| 1.1 IA symbolique | Règles à la main + comptage | mots-clés écrits par nous | Lisible, mais rate les mots manquants et les cas ambigus |
| 1.2 ML statistique | Régression logistique apprise | mots-clés (plus riches) | Apprend les poids depuis les données, corrige seule une partie des erreurs |
| 1.3 Deep Learning | Réseau de neurones | **tout le texte brut** | Fabrique ses propres features, attrape les cas ambigus |
| 1.4 Transformer / LLM | Pré-entraînement massif + attention | texte brut + sens | Comprend les paraphrases, sans entraînement sur nos phrases |
| 1.6 LLM génératif | Le transformer en mode décodeur | texte brut | Génère et explique, mais peut halluciner |

Sur des phrases **nouvelles** (1.5), l'ordre est net : **transformer ≥ réseau ≥ régression ≥ règles**. Chaque génération comprend un peu mieux le **sens**, au-delà des mots exacts.

→ Direction le **module 2** : les concepts ML (features, split, overfitting, métriques) qui structurent **tout** ça, y compris ce qui se passe dans le ventre d'un LLM.